# Notebook 03: Fine-Tuning Runs

**Goal:** Fine-tune on Python code (primary) and TinyStories prose (mandatory control).

**Outputs:** `checkpoints/code_seed*/`, `checkpoints/prose_seed*/`

**Runtime:** ~80 minutes per condition on Colab T4 GPU.

> **Note:** Both conditions are required. The prose control is mandatory for any domain-shift claim.

In [ ]:
import os
import sys
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# Define repository information
repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_url = "https://github.com/Mattral/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"  # Standard Colab clone location

# Clone the repository if it doesn't exist
if not os.path.exists(repo_path):
    print(f"Cloning {repo_url} to {repo_path}...")
    !git clone {repo_url} {repo_path}

# Change current working directory to the repository root
# This allows relative imports (like 'src.model...') to work correctly
if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

# Add the current directory (repo root) to sys.path if not already there
# This ensures 'src' is discoverable for imports.
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Install project dependencies from requirements.txt
# This ensures all necessary libraries, including transformer_lens and transformers,
# are installed with the versions specified by the project.
print(f"Installing dependencies from {repo_path}/requirements.txt...")
!pip install -r requirements.txt

# Original imports
from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

### ⚠️ Restart Runtime Required
Please go to **Runtime -> Restart session** now. After the session restarts, run the cell below to load the modules and continue.

In [1]:
# After restarting the runtime, re-run this cell to continue with the imports and model setup.
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# The current working directory should already be set to the repo root from the previous cell.
# Add the current directory (repo root) to sys.path if not already there, in case of a fresh restart.
import sys
import os

repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"

if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Changing current directory to /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning
Using device: cpu


In [2]:
import sys; sys.path.insert(0, '..')
import torch
from src.model.config import ModelConfig, TrainConfig
from src.model.finetune import run_finetuning
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
model_config = ModelConfig()

Device: cpu


In [3]:
# --- Code fine-tuning (primary condition, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Code, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'code_seed{seed}', device=device, prose_control=False)
    print(f'Done: {len(history)} checkpoints')

2026-06-13T06:41:28 | INFO     | src.model.train | Global seed set to 42
2026-06-13T06:41:28 | INFO     | src.model.finetune | Fine-tuning run: code_seed42 | device: cpu
2026-06-13T06:41:28 | INFO     | src.model.train | Loading model 'attn-only-2l' on device 'cpu'



=== Code, seed=42 ===


config.json: 0.00B [00:00, ?B/s]

./model_final.pth:   0%|          | 0.00/210M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

...

Loaded pretrained model attn-only-2l into HookedTransformer
Moving model to device:  cpu


2026-06-13T08:02:21 | INFO     | src.model.finetune | Baseline induction score mean: 0.0337
2026-06-13T08:03:22 | INFO     | src.circuits.patching | Circuit heads (>=0.5): [(1, 6)]
2026-06-13T08:03:22 | INFO     | src.model.finetune | Step-0 | IS_mean=0.0337 | task_loss=11.9411 | logit_diff_clean=4.6870
2026-06-13T08:03:25 | INFO     | src.model.train | Checkpoint saved step=0 -> ../checkpoints/code_seed7/step_000000.pt
2026-06-13T08:03:25 | INFO     | src.model.train | Global seed set to 7
2026-06-13T08:03:25 | INFO     | src.model.train | Training: 244 total steps, checkpoint every 100


Moving model to device:  cpu


2026-06-13T08:05:16 | INFO     | src.model.train | step=10 | loss=7.1538 | lr=1.67e-05 | tokens=20480
2026-06-13T08:06:49 | INFO     | src.model.train | step=20 | loss=6.1292 | lr=1.99e-05 | tokens=40960
2026-06-13T08:08:21 | INFO     | src.model.train | step=30 | loss=5.0706 | lr=1.97e-05 | tokens=61440
2026-06-13T08:09:55 | INFO     | src.model.train | step=40 | loss=5.7579 | lr=1.93e-05 | tokens=81920
2026-06-13T08:11:28 | INFO     | src.model.train | step=50 | loss=4.9578 | lr=1.87e-05 | tokens=102400
2026-06-13T08:13:00 | INFO     | src.model.train | step=60 | loss=5.4349 | lr=1.80e-05 | tokens=122880
2026-06-13T08:14:44 | INFO     | src.model.train | step=70 | loss=4.8465 | lr=1.71e-05 | tokens=143360
2026-06-13T08:16:17 | INFO     | src.model.train | step=80 | loss=2.8662 | lr=1.61e-05 | tokens=163840
2026-06-13T08:17:48 | INFO     | src.model.train | step=90 | loss=4.0608 | lr=1.49e-05 | tokens=184320
2026-06-13T08:19:21 | INFO     | src.model.train | step=100 | loss=3.8556 | l

Done: 2 checkpoints


In [6]:
!grep -n "prepend_bos" /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning/src/circuits/induction_score.py

8:TransformerLens prepends a BOS token by default (prepend_bos=True), which
10:We pass prepend_bos=False to run_with_cache so the sequence the model
36:    Uses prepend_bos=False to avoid BOS-token position offset that would
72:            prepend_bos=False,   # ← CRITICAL: prevents BOS position offset
87:                "Check TransformerLens version and prepend_bos behaviour."
164:            prepend_bos=False,   # ← CRITICAL: prevents BOS position offset


In [8]:
import os
from google.colab import files

# Define the base checkpoint directory
# The fine-tuning script saves to `../checkpoints` relative to the current working directory
# which is `/content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning`.
# Thus, the actual path is `/content/checkpoints`.
base_checkpoint_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'checkpoints'))

# Seeds used in the previous cell for code fine-tuning
seeds = [42, 123, 7]

print(f"Checking for checkpoints in: {base_checkpoint_dir}")

for seed in seeds:
    run_name = f'code_seed{seed}'
    checkpoint_path = os.path.join(base_checkpoint_dir, run_name)

    if os.path.exists(checkpoint_path) and os.path.isdir(checkpoint_path):
        print(f"Found checkpoint directory: {checkpoint_path}")
        zip_filename = f'{run_name}_checkpoints.zip'
        # Create a zip archive of the checkpoint directory
        !zip -r {zip_filename} {checkpoint_path} > /dev/null
        print(f"Downloading {zip_filename}...")
        files.download(zip_filename)
        print(f"Downloaded {zip_filename}.")
    else:
        print(f"Checkpoint directory not found for {run_name} at {checkpoint_path}. Skipping download.")

print("Finished checking and downloading code fine-tuning checkpoints.")

Checking for checkpoints in: /content/checkpoints
Found checkpoint directory: /content/checkpoints/code_seed42


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded code_seed42_checkpoints.zip.
Found checkpoint directory: /content/checkpoints/code_seed123


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded code_seed123_checkpoints.zip.
Found checkpoint directory: /content/checkpoints/code_seed7


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded code_seed7_checkpoints.zip.
Finished checking and downloading code fine-tuning checkpoints.


In [ ]:
# --- Prose control (mandatory, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Prose, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'prose_seed{seed}', device=device, prose_control=True)
    print(f'Done: {len(history)} checkpoints')

2026-06-13T09:20:39 | INFO     | src.model.train | Global seed set to 42
2026-06-13T09:20:39 | INFO     | src.model.finetune | Fine-tuning run: prose_seed42 | device: cpu
2026-06-13T09:20:39 | INFO     | src.model.train | Loading model 'attn-only-2l' on device 'cpu'



=== Prose, seed=42 ===


2026-06-13T09:20:41 | INFO     | src.model.train | Model loaded: 2L 8H d_model=512


Loaded pretrained model attn-only-2l into HookedTransformer
Moving model to device:  cpu


2026-06-13T09:20:53 | INFO     | src.model.finetune | Baseline induction score mean: 0.0336
2026-06-13T09:21:54 | INFO     | src.circuits.patching | Circuit heads (>=0.5): [(1, 6)]
2026-06-13T09:21:54 | INFO     | src.model.finetune | Step-0 | IS_mean=0.0336 | task_loss=11.7863 | logit_diff_clean=4.8071
2026-06-13T09:22:02 | INFO     | src.model.train | Checkpoint saved step=0 -> ../checkpoints/prose_seed42/step_000000.pt
2026-06-13T09:22:02 | INFO     | src.model.train | Global seed set to 42
2026-06-13T09:22:02 | INFO     | src.model.train | Training: 244 total steps, checkpoint every 100


Moving model to device:  cpu


2026-06-13T09:23:34 | INFO     | src.model.train | step=10 | loss=7.1024 | lr=1.67e-05 | tokens=20480
2026-06-13T09:25:02 | INFO     | src.model.train | step=20 | loss=5.9645 | lr=1.99e-05 | tokens=40960
2026-06-13T09:26:44 | INFO     | src.model.train | step=30 | loss=5.0264 | lr=1.97e-05 | tokens=61440
2026-06-13T09:28:16 | INFO     | src.model.train | step=40 | loss=5.1324 | lr=1.93e-05 | tokens=81920
2026-06-13T09:29:49 | INFO     | src.model.train | step=50 | loss=4.8591 | lr=1.87e-05 | tokens=102400
2026-06-13T09:31:24 | INFO     | src.model.train | step=60 | loss=4.3734 | lr=1.80e-05 | tokens=122880
2026-06-13T09:32:56 | INFO     | src.model.train | step=70 | loss=4.2638 | lr=1.71e-05 | tokens=143360
2026-06-13T09:34:28 | INFO     | src.model.train | step=80 | loss=4.2730 | lr=1.61e-05 | tokens=163840
2026-06-13T09:36:08 | INFO     | src.model.train | step=90 | loss=4.0165 | lr=1.49e-05 | tokens=184320


In [ ]:
import os
from google.colab import files

# Define the base checkpoint directory
# Assuming current working directory is /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning
base_checkpoint_dir = os.path.join(os.getcwd(), 'checkpoints')

# Seeds used in the previous cell for prose fine-tuning
seeds = [42, 123, 7]

print(f"Checking for checkpoints in: {base_checkpoint_dir}")

for seed in seeds:
    run_name = f'prose_seed{seed}' # Changed from 'code_seed' to 'prose_seed'
    checkpoint_path = os.path.join(base_checkpoint_dir, run_name)

    if os.path.exists(checkpoint_path) and os.path.isdir(checkpoint_path):
        print(f"Found checkpoint directory: {checkpoint_path}")
        zip_filename = f'{run_name}_checkpoints.zip'
        # Create a zip archive of the checkpoint directory
        !zip -r {zip_filename} {checkpoint_path} > /dev/null
        print(f"Downloading {zip_filename}...")
        files.download(zip_filename)
        print(f"Downloaded {zip_filename}.")
    else:
        print(f"Checkpoint directory not found for {run_name} at {checkpoint_path}. Skipping download.")

print("Finished checking and downloading prose fine-tuning checkpoints.")